# 06 — PL/pgSQL: небезопасный EXECUTE

> **`vuln_class`:** `PLPGSQL_UNSAFE` · **Риск:** 9/10 · **CWE-89** · **CAPEC-66**
> **Бонусный класс ТЗ (+10 баллов)** — см. [docs/adr/0010-plpgsql-bonus-path.md](../docs/adr/0010-plpgsql-bonus-path.md).

Внутри хранимой функции PL/pgSQL **динамический SQL** собирается через **конкатенацию** (`||`) или `format()` с `%s` (вместо безопасного `%L`/`%I` или `USING`). Это **SQL Injection** внутри хранимки — особенно опасный, если функция `SECURITY DEFINER`.


## 🧒 Аналогия для ребёнка

Хранимая функция — это **робот** в БД, который умеет выполнять
задания. У тебя есть запечатанный конверт с инструкцией:
«иди к **полке X** и принеси книгу».

- **Плохо:** «полку X» ты пишешь в инструкции **обычным
  фломастером поверх**. Кто-то может стереть и написать
  «иди к складу и принеси всё».
- **Хорошо:** в инструкции есть **специальный пустой квадратик**
  для номера полки, и ты вставляешь туда **табличку с числом**.
  Стереть и переписать не получится — это другой формат.

В PL/pgSQL «фломастер» — это `||` (склейка строк). «Табличка» —
это `USING $1` или `format(..., %L, x)`.


## ⚠️ Дисклеймер

SQLite не понимает PL/pgSQL. Мы **симулируем** хранимую функцию
Python-обёрткой, которая принимает «тело функции» как строку и
внутри собирает SQL — точно как PL/pgSQL.


## 1. Setup


In [ ]:
"""
@brief Подготовка окружения и mock-БД через in-memory SQLite.
@details
    Никаких внешних зависимостей кроме stdlib + sqlite3 (есть в Colab из коробки).
    SQLite используем как «упрощённую модель PostgreSQL» — он умеет
    почти весь стандартный SQL, что достаточно для демонстраций уязвимостей.
@note
    Реальная система работает на PostgreSQL (см. ADR-0001),
    использует pglast для AST-парсинга. Здесь, для наглядности,
    эмулируем аудитор через `re` (регулярки) и простой pattern matching.
"""
import sqlite3
import re
import time
from textwrap import dedent


def section(title):
    """@brief Печатает заголовок секции."""
    print("\n" + "=" * 72)
    print(title)
    print("=" * 72)


def show_result(rows, max_rows=10):
    """@brief Печатает результаты запроса в виде таблицы."""
    if not rows:
        print("  (нет строк)")
        return
    for i, r in enumerate(rows[:max_rows]):
        print(f"  {i + 1:>3}. {r}")
    if len(rows) > max_rows:
        print(f"  ... ещё {len(rows) - max_rows} строк")


def print_finding(f):
    """@brief Красиво печатает Finding от нашего аудитора."""
    print(f"  ⚠️  {f['rule_id']}")
    print(f"      vuln_class:  {f['vuln_class']}")
    print(f"      severity:    {f['severity']}")
    print(f"      risk_score:  {f['risk_score']}/10")
    print(f"      message:     {f['message']}")
    if f.get("evidence_refs"):
        print(f"      ссылки:      {', '.join(f['evidence_refs'])}")


def setup_users():
    conn = sqlite3.connect(":memory:")
    conn.execute("CREATE TABLE users (id INTEGER PRIMARY KEY, login TEXT, password TEXT)")
    conn.executemany(
        "INSERT INTO users (login, password) VALUES (?, ?)",
        [("admin", "Sup3rS3cr3t"), ("bob", "qwerty"), ("alice", "12345")],
    )
    conn.commit()
    return conn


conn = setup_users()
show_result(conn.execute("SELECT * FROM users").fetchall())


## 2. Уязвимая «хранимка» — конкатенация в EXECUTE


In [ ]:
##
# @brief Имитация уязвимой PL/pgSQL функции:
#   CREATE FUNCTION find_user(login text) RETURNS SETOF users AS $$
#   BEGIN
#       RETURN QUERY EXECUTE 'SELECT * FROM users WHERE login = ''' || login || '''';
#   END $$ LANGUAGE plpgsql;
# @warning  Конкатенация → классический SQLi.
def find_user_BAD_concat(conn, login: str):
    # Внутри «хранимки» собираем SQL через ||
    dynamic_sql = "SELECT * FROM users WHERE login = '" + login + "'"
    print(f"  EXECUTE: {dynamic_sql}")
    return conn.execute(dynamic_sql).fetchall()


##
# @brief Имитация уязвимой PL/pgSQL функции через format() c %s:
#   RETURN QUERY EXECUTE format('SELECT * FROM users WHERE login = %s', login);
# @warning  %s — НЕ безопасно (это эквивалент ||). Безопасно — %L.
def find_user_BAD_format(conn, login: str):
    dynamic_sql = "SELECT * FROM users WHERE login = %s" % login  # 🚨 %s, не %L!
    print(f"  EXECUTE: {dynamic_sql}")
    return conn.execute(dynamic_sql).fetchall()


section("Нормальный вызов — login приходит без кавычек, БД ищет 'admin'")
show_result(find_user_BAD_concat(conn, "admin"))


## 3. Атака — payload в параметре функции


In [ ]:
section("АТАКА на конкатенацию")
payload = "x' OR '1'='1"
print(f"  Атакующий: login = {payload!r}")
rows = find_user_BAD_concat(conn, payload)
print(f"  💀 {len(rows)} строк (всех пользователей с паролями):")
show_result(rows)


## 4. Аудитор Phase 1 — правила R012, R013

Phase 1 в проде использует `pglast.parse_plpgsql()` (это редкая
возможность — `sqlglot` PL/pgSQL не парсит). Здесь — regex по DDL.


In [ ]:
##
# @brief R012 — конкатенация через || внутри EXECUTE.
def audit_R012_concat(plpgsql_text):
    if re.search(r"EXECUTE\s+['\"].*?\|\|", plpgsql_text, re.IGNORECASE | re.DOTALL):
        return [{
            "rule_id":       "R012-plpgsql-execute-concat",
            "vuln_class":    "PLPGSQL_UNSAFE",
            "severity":      "high", "risk_score": 8,
            "message":       "EXECUTE с конкатенацией через || — SQL Injection",
            "evidence_refs": ["CWE-89", "CAPEC-66"],
        }]
    return []


##
# @brief R013 — format() с %s вместо %L/%I (или без USING).
def audit_R013_format_percent_s(plpgsql_text):
    if re.search(r"EXECUTE\s+format\s*\(\s*['\"].*?%s.*?['\"]",
                 plpgsql_text, re.IGNORECASE | re.DOTALL):
        return [{
            "rule_id":       "R013-plpgsql-format-without-using",
            "vuln_class":    "PLPGSQL_UNSAFE",
            "severity":      "high", "risk_score": 7,
            "message":       "format() с %s — эквивалент конкатенации, ожидается %L или USING",
            "evidence_refs": ["CWE-89"],
        }]
    return []


bad_plpgsql = """
CREATE OR REPLACE FUNCTION find_user(login text)
RETURNS SETOF users LANGUAGE plpgsql AS $$
BEGIN
  RETURN QUERY EXECUTE 'SELECT * FROM users WHERE login = ''' || login || '''';
END $$;
"""

bad_plpgsql_format = """
CREATE OR REPLACE FUNCTION find_user(login text)
RETURNS SETOF users LANGUAGE plpgsql AS $$
BEGIN
  RETURN QUERY EXECUTE format('SELECT * FROM users WHERE login = %s', login);
END $$;
"""

good_plpgsql = """
CREATE OR REPLACE FUNCTION find_user(login text)
RETURNS SETOF users LANGUAGE plpgsql AS $$
BEGIN
  RETURN QUERY EXECUTE 'SELECT * FROM users WHERE login = $1' USING login;
END $$;
"""

section("Аудитор: версия с ||")
for f in audit_R012_concat(bad_plpgsql):
    print_finding(f)

section("Аудитор: версия с format(%s)")
for f in audit_R013_format_percent_s(bad_plpgsql_format):
    print_finding(f)

section("Аудитор: безопасная версия (USING)")
fs = audit_R012_concat(good_plpgsql) + audit_R013_format_percent_s(good_plpgsql)
if fs:
    for f in fs:
        print_finding(f)
else:
    print("  ✅ Уязвимостей не найдено.")


## 5. Безопасная версия


In [ ]:
print("Эталонный fix:")
print(good_plpgsql)
print()
print("Три безопасных способа динамического SQL в PL/pgSQL:")
print("  1. EXECUTE '... = $1' USING var       — параметризация")
print("  2. EXECUTE format('... = %L', var)    — для литералов")
print("  3. EXECUTE format('... %I ...', tbl)  — для идентификаторов")
print("  ❌ EXECUTE '...' || var               — НЕЛЬЗЯ")
print("  ❌ EXECUTE format('... = %s', var)    — НЕЛЬЗЯ (%s = ||)")


## Итог

Мы увидели одно и то же на двух функциях:

- **Уязвимая** — украли данные / повредили БД / поднялись в правах.
- **Безопасная** — та же атака уходит в пустоту.

Между ними — **один аудитор** с конкретным правилом, которое можно
запустить детерминированно (без LLM) на каждом сгенерированном SQL.

## Куда дальше

- **Описание уязвимости (под микроскопом):** [problems/vulnerabilities/06-plpgsql-unsafe-execute/README.md](../../problems/vulnerabilities/06-plpgsql-unsafe-execute/README.md)
- **Варианты решения + почему так:** [problems/vulnerabilities/06-plpgsql-unsafe-execute/solutions.md](../../problems/vulnerabilities/06-plpgsql-unsafe-execute/solutions.md)
- **Архитектура цикла:** [docs/adr/0002-loop-architecture-langgraph.md](../../docs/adr/0002-loop-architecture-langgraph.md)
- **Гибридный аудитор (pglast + LLM):** [docs/adr/0004-hybrid-auditor-ast-plus-llm.md](../../docs/adr/0004-hybrid-auditor-ast-plus-llm.md)
